# Baseline Model Training

This notebook trains simple baseline pipelines for three regression targets and one classification target using the same input columns for every model.

## Targets

1. `expected_delay_days`
2. `adjusted_eta_days`
3. `freight_cost_index`
4. `delay_class`

All saved artifacts are exported to the `models/baseline/` folder as fitted scikit-learn pipelines.

## Training Plan

1. Load the processed synthetic dataset.
2. Use one shared feature set across every target.
3. Apply the same preprocessing strategy inside each pipeline.
4. Train simple baseline models.
5. Save fitted pipelines and training metadata.

In [8]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
DATA_PATH = Path("../data/processed/synthetic_shipments.csv")
MODELS_DIR = Path("../models/baseline")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (12000, 24)


,shipment_id,departure_timestamp,origin_port,destination_port,origin_region,destination_region,route_corridor,ocean_side,vessel_type,cargo_type,...,port_handling_days,geopolitical_event,geopolitical_delay_days,expected_delay_days,adjusted_eta_days,risk_score,freight_cost_index,baseline_arrival_timestamp,adjusted_arrival_timestamp,delay_class
0,SHP-0000001,2025-10-30 22:00:00,Shanghai,Houston,East Asia,North America,Pacific Crossing,Pacific Side,Bulk Carrier,Electronics,...,0.89,NaN,0.00,1.42,22.39,0.269,1.028,2025-11-20 21:15:24.026740490,2025-11-22 07:25:06.079515185,on_time
1,SHP-0000002,2025-02-05 20:00:00,Singapore,Santos,Southeast Asia,South America,Cape Diversion,Indian-Atlantic Side,Tanker,Textiles,...,0.65,NaN,0.00,0.99,24.13,0.232,1.020,2025-02-28 23:31:36.271944048,2025-03-01 23:11:25.236060987,on_time
2,SHP-0000003,2025-08-13 14:00:00,Shanghai,Los Angeles,East Asia,North America,Pacific Crossing,Pacific Side,Container,Auto Parts,...,1.47,Customs Slowdown,0.53,2.73,13.80,0.417,1.078,2025-08-24 15:38:03.028058946,2025-08-27 09:15:54.984031888,delayed
3,SHP-0000004,2025-08-05 07:00:00,Sydney,Shanghai,Oceania,East Asia,Cape Diversion,Indian-Atlantic Side,Bulk Carrier,Chemicals,...,1.72,NaN,0.00,1.72,15.69,0.293,1.034,2025-08-19 06:20:09.168756973,2025-08-20 23:32:00.118745810,delayed
4,SHP-0000005,2025-08-27 22:00:00,Houston,Sydney,North America,Oceania,Cape Diversion,Indian-Atlantic Side,Container,Food,...,1.29,NaN,0.00,1.29,22.64,0.257,1.026,2025-09-18 06:32:50.372501524,2025-09-19 13:26:36.684039105,on_time


## Shared Feature Set

The same raw input columns are used for all four pipelines so future user-facing simulations can reuse a single input schema.

In [10]:
BASE_FEATURES = [
    "departure_timestamp",
    "origin_port",
    "destination_port",
    "origin_region",
    "destination_region",
    "route_corridor",
    "ocean_side",
    "vessel_type",
    "cargo_type",
    "distance_nm",
    "avg_speed_knots",
    "baseline_eta_days",
    "weather_delay_days",
    "port_handling_days",
    "geopolitical_event",
    "geopolitical_delay_days",
]

REGRESSION_TARGETS = [
    "expected_delay_days",
    "adjusted_eta_days",
    "freight_cost_index",
]
CLASSIFICATION_TARGET = "delay_class"

NUMERIC_FEATURES = [
    "distance_nm",
    "avg_speed_knots",
    "baseline_eta_days",
    "weather_delay_days",
    "port_handling_days",
    "geopolitical_delay_days",
    "departure_month",
    "departure_dayofweek",
    "departure_hour",
]

CATEGORICAL_FEATURES = [
    "origin_port",
    "destination_port",
    "origin_region",
    "destination_region",
    "route_corridor",
    "ocean_side",
    "vessel_type",
    "cargo_type",
    "geopolitical_event",
]

def add_time_features(frame):
    enriched = frame.copy()
    departure_ts = pd.to_datetime(enriched["departure_timestamp"], errors="coerce")
    enriched["departure_month"] = departure_ts.dt.month.fillna(0).astype(int)
    enriched["departure_dayofweek"] = departure_ts.dt.dayofweek.fillna(0).astype(int)
    enriched["departure_hour"] = departure_ts.dt.hour.fillna(0).astype(int)
    return enriched.drop(columns=["departure_timestamp"])

X = df[BASE_FEATURES].copy()
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[CLASSIFICATION_TARGET],
)

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]

print("Shared input columns:")
print(BASE_FEATURES)

Shared input columns:
['departure_timestamp', 'origin_port', 'destination_port', 'origin_region', 'destination_region', 'route_corridor', 'ocean_side', 'vessel_type', 'cargo_type', 'distance_nm', 'avg_speed_knots', 'baseline_eta_days', 'weather_delay_days', 'port_handling_days', 'geopolitical_event', 'geopolitical_delay_days']


In [11]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

def build_pipeline(model):
    return Pipeline(steps=[
        ("time_features", FunctionTransformer(add_time_features, validate=False)),
        ("preprocessor", preprocessor),
        ("model", model),
    ])

## Regression Pipelines

A simple Ridge regressor is trained for each continuous target using the same feature set and preprocessing steps.

In [12]:
regression_results = []
saved_model_paths = {}

for target in REGRESSION_TARGETS:
    y_train = df.loc[train_idx, target]
    y_test = df.loc[test_idx, target]

    pipeline = build_pipeline(Ridge(alpha=1.0))
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    metrics = {
        "target": target,
        "mae": round(mean_absolute_error(y_test, predictions), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_test, predictions))), 4),
        "r2": round(r2_score(y_test, predictions), 4),
    }
    regression_results.append(metrics)

    model_path = MODELS_DIR / f"{target}_pipeline.joblib"
    dump(pipeline, model_path)
    saved_model_paths[target] = str(model_path.resolve())

pd.DataFrame(regression_results)

,target,mae,rmse,r2
0,expected_delay_days,0.0025,0.0047,1.0000
1,adjusted_eta_days,0.0055,0.0071,1.0000
2,freight_cost_index,0.0006,0.0010,0.9996


## Delay Class Pipeline

A logistic regression classifier is trained for `delay_class` using the same input schema as the regression pipelines.

In [13]:
y_train_cls = df.loc[train_idx, CLASSIFICATION_TARGET]
y_test_cls = df.loc[test_idx, CLASSIFICATION_TARGET]

classification_pipeline = build_pipeline(
    LogisticRegression(max_iter=2000, solver="saga", random_state=RANDOM_STATE)
)
classification_pipeline.fit(X_train, y_train_cls)
class_predictions = classification_pipeline.predict(X_test)

classification_results = pd.DataFrame([
    {
        "target": CLASSIFICATION_TARGET,
        "accuracy": round(accuracy_score(y_test_cls, class_predictions), 4),
        "macro_f1": round(f1_score(y_test_cls, class_predictions, average="macro"), 4),
    }
])

classification_model_path = MODELS_DIR / "delay_class_pipeline.joblib"
dump(classification_pipeline, classification_model_path)
saved_model_paths[CLASSIFICATION_TARGET] = str(classification_model_path.resolve())

classification_results

,target,accuracy,macro_f1
0,delay_class,0.9929,0.9873


## Save Training Metadata

The notebook also saves feature definitions, metrics, and artifact paths so downstream code can load the right pipeline with the expected input schema.

In [14]:
metadata = {
    "random_state": RANDOM_STATE,
    "base_features": BASE_FEATURES,
    "numeric_features_after_time_expansion": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "regression_targets": REGRESSION_TARGETS,
    "classification_target": CLASSIFICATION_TARGET,
    "regression_metrics": regression_results,
    "classification_metrics": classification_results.to_dict(orient="records"),
    "saved_model_paths": saved_model_paths,
}

metadata_path = MODELS_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved artifacts:")
for name, path in saved_model_paths.items():
    print(f"- {name}: {path}")
print(f"- metadata: {metadata_path.resolve()}")

display(pd.DataFrame(regression_results))
display(classification_results)

Saved artifacts:
- expected_delay_days: C:\Users\filip\GitHub\Chain-Pulse-AI\models\baseline\expected_delay_days_pipeline.joblib
- adjusted_eta_days: C:\Users\filip\GitHub\Chain-Pulse-AI\models\baseline\adjusted_eta_days_pipeline.joblib
- freight_cost_index: C:\Users\filip\GitHub\Chain-Pulse-AI\models\baseline\freight_cost_index_pipeline.joblib
- delay_class: C:\Users\filip\GitHub\Chain-Pulse-AI\models\baseline\delay_class_pipeline.joblib
- metadata: C:\Users\filip\GitHub\Chain-Pulse-AI\models\baseline\training_metadata.json


,target,mae,rmse,r2
0,expected_delay_days,0.0025,0.0047,1.0000
1,adjusted_eta_days,0.0055,0.0071,1.0000
2,freight_cost_index,0.0006,0.0010,0.9996


,target,accuracy,macro_f1
0,delay_class,0.9929,0.9873
